In [ ]:
# ==========================================
# ENVIRONMENT SETUP & INSTALLATIONS
# ==========================================
# Install data engineering libraries
%pip install pyarrow fastparquet -q

import pandas as pd
import numpy as np
import os
import json
import glob

In [ ]:
# ==========================================
# LOCATE THE PROJECT FOLDER
# ==========================================
# Finds the cloned repository (the folder that contains 03_scripts).
# Works locally, in Colab and elsewhere. Set the PROJECT_ROOT environment variable to override.
import os
from pathlib import Path

def find_project_root():
    if os.environ.get("PROJECT_ROOT"):
        return Path(os.environ["PROJECT_ROOT"])
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "03_scripts").is_dir():
            return folder
    colab_clone = Path("/content/rail-station-property-values")
    if colab_clone.is_dir():
        return colab_clone
    raise FileNotFoundError("Project folder not found. Run the notebook from inside the cloned repository, or set PROJECT_ROOT.")

PROJECT_ROOT = str(find_project_root())
print(f"Project folder: {PROJECT_ROOT}")

In [ ]:
# ==========================================
# PART 1 (RESIDENTIAL)
# PATH DEFINITIONS AND FOLDER CREATION
# ==========================================
import os

# Setup Paths
BASE_DIR = PROJECT_ROOT

# Define input paths (os.path.join safely handles the relative paths)
raw_property_path = os.path.join(BASE_DIR, "01_raw_data/property")

# Residential Valuation & Controls
hmlr_dir_path = f"{raw_property_path}/HMLand_Registry_Price_Paid_Data"
hpi_path = f"{raw_property_path}/HPI/UK-HPI-full-file-2026-02.csv"
epc_path = f"{raw_property_path}/EPC/EPC_Devon_County.csv"

# Commercial Valuation (VOA)
voa_2017_list_path = f"{raw_property_path}/VOA_Non_Domestic_Rating_Data/2017-listentries-compiled-epoch-0056-baseline.csv"
voa_2017_summary_path = f"{raw_property_path}/VOA_Non_Domestic_Rating_Data/2017-summaryvaluations-compiled-epoch-0056.csv"
voa_2023_list_path = f"{raw_property_path}/VOA_Non_Domestic_Rating_Data/2023-listentries-compiled-epoch-0020-baseline.csv"
voa_2023_summary_path = f"{raw_property_path}/VOA_Non_Domestic_Rating_Data/2023-summaryvaluations-compiled-epoch-0020-baseline.csv"

# Define output paths
output_res_cleaned = os.path.join(BASE_DIR, "02_processed_data/01_cleaned/residential_cleaned.parquet")
output_com_cleaned = os.path.join(BASE_DIR, "02_processed_data/01_cleaned/commercial_cleaned.parquet")
output_attrition_log = os.path.join(BASE_DIR, "05_logs/script_logs/attrition_log.json")

# Automatically generate the required directories before exporting data
folders_to_create = [
    '02_processed_data/01_cleaned',
    '02_processed_data/02_spatial',
    '02_processed_data/03_model_ready',
    '04_results/tables',
    '04_results/figures',
    '04_results/maps',
    '05_logs/script_logs',
    '05_logs/quality_checks',
]

for folder in folders_to_create:
    folder_path = os.path.join(BASE_DIR, folder)
    os.makedirs(folder_path, exist_ok=True)

print("Cell 3 Complete: Paths defined and project folder architecture verified/created.")

Cell 3 Complete: Paths defined and project folder architecture verified/created.


In [ ]:
# ==========================================
#  PART 1 (RESIDENTIAL) - PHASE 1A PRE-CHECK: HMLR FILE STRUCTURE INSPECTION
# ==========================================
def inspect_hmlr(directory_path, num_lines=3):
    print(f"--- INSPECTING HMLR DATA IN: {os.path.basename(directory_path)} ---")
    files = glob.glob(os.path.join(directory_path, '*.csv'))
    if not files:
        print("ERROR: No files found to inspect.")
        return

    # Check the first file found
    test_file = files[0]
    print(f"Inspecting file: {os.path.basename(test_file)}")
    with open(test_file, 'r', encoding='ISO-8859-1') as f:
        for i, line in enumerate(f):
            if i >= num_lines: break
            print(f"Row {i+1}: {line.strip()}")

    print("\n[VERIFICATION]: If these rows contain raw data values (e.g., {4C1B...}) "
          "without column names, the 'header=None' parameter in your script is correct.")

inspect_hmlr(hmlr_dir_path)

--- INSPECTING HMLR DATA IN: HMLand_Registry_Price_Paid_Data_Residential_Value ---
Inspecting file: pp-2026.csv
Row 1: "{4E75C29E-2FBE-8922-E063-4804A8C094D0}","78000","2026-01-28 00:00","NR30 1DG","T","N","F","27","","MAYGROVE ROAD","","GREAT YARMOUTH","GREAT YARMOUTH","NORFOLK","B","A"
Row 2: "{4E75C29E-2FBF-8922-E063-4804A8C094D0}","260000","2026-03-11 00:00","NR2 4BJ","T","N","F","47","","WYMER STREET","","NORWICH","NORWICH","NORFOLK","B","A"
Row 3: "{4E75C29E-2FC2-8922-E063-4804A8C094D0}","120000","2026-03-26 00:00","NR31 7EX","T","N","F","5","","CAMBRIDGE AVENUE","GORLESTON","GREAT YARMOUTH","GREAT YARMOUTH","NORFOLK","B","A"

[VERIFICATION]: If these rows contain raw data values (e.g., {4C1B...}) without column names, the 'header=None' parameter in your script is correct.


In [ ]:
# ==========================================
#  PART 1 (RESIDENTIAL) - PHASE 1A: ATTRITION LOG INITIALIsATION & MEMORY-EFFICIENT HMLR INGESTION
# ==========================================
import glob
import os
import gc

# 1. Initialise the Attrition Log
attrition_log = {
    "HMLR": {},
    "VOA_2017": {},
    "VOA_2023": {},
    "EPC": {}
}
print("Initialising Memory-Efficient Data Ingestion...")

# 2. Define HMLR schema and strictly limit columns to save RAM
hmlr_columns = [
    'Transaction_ID', 'Price', 'Date_of_Transfer', 'Postcode', 'Property_Type',
    'Old_New', 'Duration', 'PAON', 'SAON', 'Street', 'Locality', 'Town_City',
    'District', 'County', 'PPD_Category_Type', 'Record_Status'
]

# We don't need 'Locality', 'Town_City', 'Duration', etc. for the regression
cols_to_keep = ['Transaction_ID', 'Price', 'Date_of_Transfer', 'Postcode',
                'Property_Type', 'Old_New', 'PAON', 'SAON', 'Street', 'District', 'County',
                'PPD_Category_Type']

target_authorities = [
    'EXETER', 'EAST DEVON', 'MID DEVON', 'NORTH DEVON',
    'SOUTH HAMS', 'TEIGNBRIDGE', 'TORRIDGE', 'WEST DEVON',
    'PLYMOUTH', 'CITY OF PLYMOUTH', 'TORBAY'
]

# 3. Memory-Optimised Ingestion Loop
print(f"Searching for CSVs in: {hmlr_dir_path}")

# Instructs Python to read ALL .csv files in the directory
hmlr_files = glob.glob(os.path.join(hmlr_dir_path, '*.csv'))

if not hmlr_files:
    print(f"ERROR: No CSV files found in {hmlr_dir_path}. Please check your BASE_DIR and folder structure.")
else:
    print(f"Found {len(hmlr_files)} CSV files. Beginning iterative extraction...")

    df_devon_list = []
    total_national_rows = 0

    for file in hmlr_files:
        # print(f"Processing {os.path.basename(file)}...")

        # Read just one year into memory, and only the columns we need
        temp_df = pd.read_csv(file, header=None, names=hmlr_columns, usecols=cols_to_keep, low_memory=False)

        # Log the national row count before we filter
        total_national_rows += len(temp_df)

        # Filter for Devon to drop millions of rows from RAM
        devon_chunk = temp_df[
            temp_df['District'].isin(target_authorities) |
            temp_df['County'].isin(['DEVON', 'PLYMOUTH', 'CITY OF PLYMOUTH', 'TORBAY'])
        ].copy()

        df_devon_list.append(devon_chunk)

        # Delete the massive national chunk from memory and clear the garbage collector
        del temp_df
        gc.collect()

    # Combine only the lightweight Devon data
    df_devon_hmlr = pd.concat(df_devon_list, ignore_index=True)

    # Update the Attrition Log
    attrition_log["HMLR"]["01_Raw_National_Rows"] = total_national_rows
    attrition_log["HMLR"]["02_Devon_Filtered_Rows"] = len(df_devon_hmlr)

    print(f"\nCell Complete: Successfully extracted Devon data from {len(hmlr_files)} years.")
    print(f"Total Starting HMLR Rows (National): {total_national_rows:,}")
    print(f"Rows retained for Devon study area: {len(df_devon_hmlr):,}")

Initialising Memory-Efficient Data Ingestion...
Searching for CSVs in: /content/drive/MyDrive/MSc_Dissertation_Project/01_raw_data/property/HMLand_Registry_Price_Paid_Data_Residential_Value
Found 9 CSV files. Beginning iterative extraction...

Cell Complete: Successfully extracted Devon data from 9 years.
Total Starting HMLR Rows (National): 8,043,771
Rows retained for Devon study area: 193,671


In [ ]:
# ==========================================
#  PART 1 (RESIDENTIAL) - PHASE 1B PRE-CHECK: HPI FILE STRUCTURE INSPECTION
# ==========================================
def inspect_hpi(path, num_lines=10):
    print(f"--- INSPECTING HPI DATA: {os.path.basename(path)} ---")
    try:
        with open(path, 'r', encoding='ISO-8859-1') as f:
            for i, line in enumerate(f):
                if i >= num_lines: break
                print(f"Row {i+1}: {line.strip()}")

        print("\n[VERIFICATION]: Check if the header row (e.g., 'Date', 'RegionName', 'Index') "
              "is on the very first line.")
    except Exception as e:
        print(f"ERROR: Could not read HPI file: {e}")

# Run the inspection
inspect_hpi(hpi_path)

--- INSPECTING HPI DATA: UK-HPI-full-file-2026-02.csv ---
Row 1: Date,RegionName,AreaCode,AveragePrice,Index,IndexSA,1m%Change,12m%Change,AveragePriceSA,SalesVolume,DetachedPrice,DetachedIndex,Detached1m%Change,Detached12m%Change,SemiDetachedPrice,SemiDetachedIndex,SemiDetached1m%Change,SemiDetached12m%Change,TerracedPrice,TerracedIndex,Terraced1m%Change,Terraced12m%Change,FlatPrice,FlatIndex,Flat1m%Change,Flat12m%Change,CashPrice,CashIndex,Cash1m%Change,Cash12m%Change,CashSalesVolume,MortgagePrice,MortgageIndex,Mortgage1m%Change,Mortgage12m%Change,MortgageSalesVolume,FTBPrice,FTBIndex,FTB1m%Change,FTB12m%Change,FOOPrice,FOOIndex,FOO1m%Change,FOO12m%Change,NewPrice,NewIndex,New1m%Change,New12m%Change,NewSalesVolume,OldPrice,OldIndex,Old1m%Change,Old12m%Change,OldSalesVolume
Row 2: 01/01/2004,Aberdeenshire,S12000034,84638,41.1,,,,,388,130620,43.2,,,73972,40.9,,,58247,38.8,,,49322,45.7,,,,,,,,,,,,,,,,,,,,,112843,40.7,,,103,81273,41,,,285
Row 3: 01/02/2004,Aberdeenshire,S12000034,84623,41

In [ ]:
# ==========================================
#  PART 1 (RESIDENTIAL) - PHASE 1B: DEDUPLICATION & PRICE NORMALISATION
# ==========================================
import json
import pandas as pd
import numpy as np

print("Initiating Price Normalisation")

# Define the Robustness Variable (New Build)
df_devon_hmlr['Is_New_Build'] = df_devon_hmlr['Old_New'].map({'Y': 1, 'N': 0})

# Time-Series Formatting for the HPI Merge
df_devon_hmlr['Date_of_Transfer'] = pd.to_datetime(df_devon_hmlr['Date_of_Transfer'], errors='coerce')
df_devon_hmlr['Join_Date'] = df_devon_hmlr['Date_of_Transfer'].dt.to_period('M').astype(str)

# Load ONS House Price Index (HPI)
df_hpi = pd.read_csv(hpi_path, low_memory=False)
df_hpi_regional = df_hpi[df_hpi['RegionName'].str.upper() == 'SOUTH WEST'].copy()

# Explicitly stating the UK Date Format
df_hpi_regional['Date'] = pd.to_datetime(df_hpi_regional['Date'], format='%d/%m/%Y', errors='coerce')

df_hpi_regional['Join_Date'] = df_hpi_regional['Date'].dt.to_period('M').astype(str)
hpi_lookup = df_hpi_regional[['Join_Date', 'Index']].rename(columns={'Index': 'HPI_Value'})

# Ensure HPI_Value is numeric
hpi_lookup['HPI_Value'] = pd.to_numeric(hpi_lookup['HPI_Value'], errors='coerce')

# Extract Base Value & Merge
hpi_lookup = hpi_lookup.sort_values('Join_Date')
latest_hpi_value = hpi_lookup['HPI_Value'].iloc[-1]
df_devon_hmlr = df_devon_hmlr.merge(hpi_lookup, on='Join_Date', how='left')

# Drop anomalous rows and update Log
df_devon_hmlr = df_devon_hmlr.dropna(subset=['HPI_Value'])
attrition_log["HMLR"]["03_HPI_Matched_Rows"] = len(df_devon_hmlr)

# Retain only Category A transactions (full open-market sales).
# Category B records (e.g. repossessions, buy-to-let/portfolio transfers, transfers to
# non-private individuals) can transact below open-market value and are excluded as
# standard hedonic practice. PPD_Category_Type is dropped before save by the keep-list
# below, so scripts 2-4 are unaffected: they simply receive a smaller, cleaner sample.
pre_category_count = len(df_devon_hmlr)
df_devon_hmlr = df_devon_hmlr[
    df_devon_hmlr['PPD_Category_Type'].astype(str).str.strip().str.upper() == 'A'
].copy()
attrition_log["HMLR"]["03a_Category_A_Filtered_Rows"] = len(df_devon_hmlr)
print(f"Category A filter: retained {len(df_devon_hmlr):,} of {pre_category_count:,} "
      f"transactions (dropped {pre_category_count - len(df_devon_hmlr):,} Category B).")

# Execute Normalisation Formula
df_devon_hmlr['Price_Normalised'] = df_devon_hmlr['Price'] * (latest_hpi_value / df_devon_hmlr['HPI_Value'])

# Natural log of the normalised price (dependent variable for the log-log hedonic model).
df_devon_hmlr['Log_Price'] = np.log(df_devon_hmlr['Price_Normalised'])

## Remove market Outliers
print("\nApplying Outlier Filters...")
initial_filter_count = len(df_devon_hmlr)

lower_bound = 50000
upper_bound = df_devon_hmlr['Price'].quantile(0.999)

# Identify outliers for the quality checks log
outliers = df_devon_hmlr[(df_devon_hmlr['Price'] < lower_bound) | (df_devon_hmlr['Price'] > upper_bound)]

if not outliers.empty:
    drop_log_path = os.path.join(PROJECT_ROOT, '05_logs/quality_checks/dropped_outliers_residential.csv')
    outliers.to_csv(drop_log_path, index=False)
    print(f"Flagged {len(outliers)} outliers (<£{lower_bound} or >£{upper_bound:,.0f}). Saved to quality_checks/.")

# Apply the filter to keep only genuine open-market transactions
df_devon_hmlr = df_devon_hmlr[
    (df_devon_hmlr['Price'] >= lower_bound) &
    (df_devon_hmlr['Price'] <= upper_bound)
].copy()

attrition_log["HMLR"]["03b_Outlier_Filtered_Rows"] = len(df_devon_hmlr)

# Deduplicate Transactions
initial_dedup_count = len(df_devon_hmlr)
df_devon_hmlr = df_devon_hmlr.drop_duplicates(subset=['Transaction_ID'], keep='first').copy()
df_devon_hmlr = df_devon_hmlr.reset_index(drop=True)

attrition_log["HMLR"]["04_De-duplicated_Rows"] = len(df_devon_hmlr)
print(f"Deduplication complete: Removed {initial_dedup_count - len(df_devon_hmlr)} duplicates.")

# Memory Cleanup
cols_to_keep = ['Transaction_ID', 'Price', 'Price_Normalised', 'Log_Price', 'Date_of_Transfer',
                'Postcode', 'Property_Type', 'Is_New_Build', 'PAON', 'SAON', 'Street']
df_devon_hmlr = df_devon_hmlr[cols_to_keep]

# Save to Parquet
df_devon_hmlr.to_parquet(output_res_cleaned, index=False)

# Save Attrition Log
with open(output_attrition_log, 'w') as f:
    json.dump(attrition_log, f, indent=4)

print(f"\nCell 5 Complete: Normalized & Cleaned dataset saved. Final count: {len(df_devon_hmlr):,} residential rows.")

Initiating Price Normalisation
Category A filter: retained 165,929 of 193,102 transactions (dropped 27,173 Category B).

Applying Outlier Filters...
Flagged 315 outliers (<£50000 or >£1,950,000). Saved to quality_checks/.
Deduplication complete: Removed 0 duplicates.

Cell 5 Complete: Normalized & Cleaned dataset saved. Final count: 165,614 residential rows.


In [ ]:
# ==========================================
# PART 1 (RESIDENTIAL) - PHASE 1B POST-CHECK: DATA INSPECTION
# ==========================================
# Display the column headers and first 10 rows
print("Dataframe Head (First 10 Rows)")
display(df_devon_hmlr.head(10))

# Print information about the dataframe (check data types and non-null counts)
print("\n--- Dataframe Info ---")
df_devon_hmlr.info()

# Check the value counts for the new robustness variable
print("\n--- Distribution of 'Is_New_Build' ---")
print(df_devon_hmlr['Is_New_Build'].value_counts())

Dataframe Head (First 10 Rows)


,Transaction_ID,Price,Price_Normalised,Log_Price,Date_of_Transfer,Postcode,Property_Type,Is_New_Build,PAON,SAON,Street
0,{4C1B673F-0132-D8EA-E063-4704A8C0BF7D},270000,270000.0,12.506177,2026-01-16,PL9 7RQ,T,0,SCOTTS COTTAGE,2,MILLWAY PLACE
1,{4C1B673F-025A-D8EA-E063-4704A8C0BF7D},199500,199500.0,12.203570,2026-01-09,PL1 3QP,F,0,"LOWER GROUND FLOOR FLAT, 121",NaN,DURNFORD STREET
2,{4C1B673F-025D-D8EA-E063-4704A8C0BF7D},272000,272000.0,12.513557,2026-01-30,EX10 9TP,T,0,19A,NaN,TYRRELL MEAD
3,{4C1B673F-025E-D8EA-E063-4704A8C0BF7D},485000,485000.0,13.091904,2026-01-29,PL6 5PU,D,0,25,NaN,WARDLOW GARDENS
4,{4C1B673F-0260-D8EA-E063-4704A8C0BF7D},390000,390000.0,12.873902,2026-02-13,TQ11 0DB,T,0,114,NaN,PLYMOUTH ROAD
5,{4C1B673F-0261-D8EA-E063-4704A8C0BF7D},237500,237500.0,12.377923,2026-02-02,TQ2 7RS,S,0,5,NaN,LOWLEY BROOK COURT
6,{4C1B673F-0262-D8EA-E063-4704A8C0BF7D},140000,140000.0,11.849398,2026-01-30,PL7 1TL,T,0,22,NaN,MERAFIELD ROAD
7,{4C1B673F-0263-D8EA-E063-4704A8C0BF7D},185000,185000.0,12.128111,2026-02-16,EX17 2BP,T,0,15,NaN,PARLIAMENT STREET
8,{4C1B673F-0265-D8EA-E063-4704A8C0BF7D},515000,515000.0,13.151922,2026-01-23,EX13 7BR,D,0,FOXGROVE,NaN,NaN
9,{4C1B673F-0269-D8EA-E063-4704A8C0BF7D},405000,405000.0,12.911642,2026-02-12,PL7 1YW,D,0,19,NaN,GREENLEES DRIVE



--- Dataframe Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 165614 entries, 0 to 165613
Data columns (total 11 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   Transaction_ID    165614 non-null  object        
 1   Price             165614 non-null  int64         
 2   Price_Normalised  165614 non-null  float64       
 3   Log_Price         165614 non-null  float64       
 4   Date_of_Transfer  165614 non-null  datetime64[ns]
 5   Postcode          165578 non-null  object        
 6   Property_Type     165614 non-null  object        
 7   Is_New_Build      165614 non-null  int64         
 8   PAON              165614 non-null  object        
 9   SAON              20625 non-null   object        
 10  Street            155614 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(2), object(6)
memory usage: 13.9+ MB

--- Distribution of 'Is_New_Build' ---
Is_New_Build
0    148788
1  

In [ ]:
# ==========================================
#  PART 1 (RESIDENTIAL) - PHASE 1B POST-CHECK: RESIDENTIAL PARQUET DATA INSPECTION
# ==========================================
import pandas as pd

# Use the full path from the root directory for the residential file
file_path = os.path.join(PROJECT_ROOT, '02_processed_data/01_cleaned/residential_cleaned.parquet')

# Load the file
df_check = pd.read_parquet(file_path)

# Verify success
print(f"Successfully loaded {len(df_check)} rows.")

# Print shape (rows, columns)
print(f"Total Dimensions: {df_check.shape[0]} rows and {df_check.shape[1]} columns.")

# Display the first 5 rows to check structure
print("\nFirst 5 rows of the dataset:")
print(df_check.head())

# Check column data types
print("\nColumn Data Types:")
print(df_check.dtypes)

# Describe the key numeric columns
print("\nSummary Statistics for Key Variables:")
print(df_check[['Price', 'Price_Normalised', 'Is_New_Build']].describe())

Successfully loaded 165614 rows.
Total Dimensions: 165614 rows and 11 columns.

First 5 rows of the dataset:
                           Transaction_ID   Price  Price_Normalised  \
0  {4C1B673F-0132-D8EA-E063-4704A8C0BF7D}  270000          270000.0   
1  {4C1B673F-025A-D8EA-E063-4704A8C0BF7D}  199500          199500.0   
2  {4C1B673F-025D-D8EA-E063-4704A8C0BF7D}  272000          272000.0   
3  {4C1B673F-025E-D8EA-E063-4704A8C0BF7D}  485000          485000.0   
4  {4C1B673F-0260-D8EA-E063-4704A8C0BF7D}  390000          390000.0   

   Log_Price Date_of_Transfer  Postcode Property_Type  Is_New_Build  \
0  12.506177       2026-01-16   PL9 7RQ             T             0   
1  12.203570       2026-01-09   PL1 3QP             F             0   
2  12.513557       2026-01-30  EX10 9TP             T             0   
3  13.091904       2026-01-29   PL6 5PU             D             0   
4  12.873902       2026-02-13  TQ11 0DB             T             0   

                           PAON  SAON

In [ ]:
# ==========================================
# PART 2 (COMMERCIAL) PRE-CHECK: PEEK AT RAW VOA DATA
# ==========================================
def peek_file(path, num_lines=5):
    print(f"--- Peeking at: {os.path.basename(path)} ---")
    with open(path, 'r', encoding='ISO-8859-1') as f:
        for i, line in enumerate(f):
            if i >= num_lines: break
            print(f"Line {i+1}: {line.strip()}")
            print("-" * 20)

# Peek at the 2017 files
peek_file(voa_2017_list_path)
peek_file(voa_2017_summary_path)

--- Peeking at: 2017-listentries-compiled-epoch-0056-baseline.csv ---
Line 1: 1*0116**00012434001011*EL*SCHOOL AND PREMISES*251031212*ST PETER & PAUL VA PRIMARY SCHOOL, ABERDEEN ROAD, BRISTOL**ST PETER & PAUL VA PRIMARY SCHOOL*ABERDEEN ROAD*BRISTOL***BS6 6HY***34500**18427888000**159G****20340499212*01-APR-2017**
--------------------
Line 2: 2*0116**00015999010012*CS*SHOP AND PREMISES*8581828000*MODERN GLASS 10, ABBOTSFORD ROAD, BRISTOL**MODERN GLASS 10*ABBOTSFORD ROAD*BRISTOL***BS6 6HB***4900**18870894000**249G****22422615212*01-APR-2017**
--------------------
Line 3: 3*0116**00015999001001*CS*SHOP AND PREMISES*4402689000*1-3, ABBOTSFORD ROAD, BRISTOL**1-3*ABBOTSFORD ROAD*BRISTOL***BS6 6EZ***9700**18870924000**249G****20727125212*01-APR-2017**
--------------------
Line 4: 4*0116**00015999005006*CS*SHOP AND PREMISES*251027212*3A & 5, ABBOTSFORD ROAD, BRISTOL**3A & 5*ABBOTSFORD ROAD*BRISTOL***BS6 6EZ***9000**18870972000**249G****20713939212*01-APR-2017**
--------------------
Line 5: 5*0

In [ ]:
# ==========================================
# PART 2 (COMMERCIAL) PRE-CHECK: COUNT DEVON ROWS IN FILES
# ==========================================
import pandas as pd

def count_devon_properties(file_path):
    print(f"--- Scanning: {os.path.basename(file_path)} ---")
    devon_prefixes = ('EX', 'TQ', 'PL')
    count = 0
    # Process in chunks to avoid memory crashes
    for chunk in pd.read_csv(file_path, sep='*', header=None, chunksize=100000,
                             dtype=str, usecols=[14], names=['Postcode'], low_memory=False):
        # Count how many rows start with Devon prefixes
        matches = chunk['Postcode'].str.startswith(devon_prefixes, na=False).sum()
        count += matches
    print(f"Total Devon properties found: {count:,}")
    return count

# Run for all four files
files_to_check = [voa_2017_list_path, voa_2023_list_path]
for f in files_to_check:
    count_devon_properties(f)

--- Scanning: 2017-listentries-compiled-epoch-0056-baseline.csv ---
Total Devon properties found: 90,704
--- Scanning: 2023-listentries-compiled-epoch-0020-baseline.csv ---
Total Devon properties found: 80,024


In [ ]:
# ==========================================
# PART 2 (COMMERCIAL) PRE-CHECK: VERIFY COLUMN POSITIONS
# ==========================================

def inspect_voa_schema(list_path, sum_path, label, n_cols=20):
    print(f"\n--- {label} Schema ---")
    df_list = next(pd.read_csv(list_path, sep='*', header=None, chunksize=1, dtype=str))
    df_sum  = next(pd.read_csv(sum_path,  sep='*', header=None, chunksize=1, dtype=str))

    print("List file columns:")
    for i, val in enumerate(df_list.iloc[0, :n_cols]):
        print(f"  [{i:02d}] {val}")

    print("Summary file columns (Line_Number='01' rows only):")
    for i, val in enumerate(df_sum.iloc[0, :n_cols]):
        print(f"  [{i:02d}] {val}")

inspect_voa_schema(voa_2017_list_path, voa_2017_summary_path, "2017")
inspect_voa_schema(voa_2023_list_path, voa_2023_summary_path, "2023")

# List:    [03]=Property_UARN  [05]=Description  [14]=Postcode  [17]=Rateable_Value  [19]=Assessment_UARN
# Summary: [01]=Assessment_UARN  [16]=Total_Floor_Area


--- 2017 Schema ---
List file columns:
  [00] 1
  [01] 0116
  [02] nan
  [03] 00012434001011
  [04] EL
  [05] SCHOOL AND PREMISES
  [06] 251031212
  [07] ST PETER & PAUL VA PRIMARY SCHOOL, ABERDEEN ROAD, BRISTOL
  [08] nan
  [09] ST PETER & PAUL VA PRIMARY SCHOOL
  [10] ABERDEEN ROAD
  [11] BRISTOL
  [12] nan
  [13] nan
  [14] BS6 6HY
  [15] nan
  [16] nan
  [17] 34500
  [18] nan
  [19] 18427888000
Summary file columns (Line_Number='01' rows only):
  [00] 01
  [01] 23435602000
  [02] 12786606000
  [03] 0335
  [04] The Occupier
  [05] 28
  [06] nan
  [07] nan
  [08] nan
  [09] GREAT HOLLANDS SQUARE
  [10] nan
  [11] BRACKNELL
  [12] BERKS
  [13] RG12 8UX
  [14] 293025
  [15] Warehouse And Premises
  [16] 100.32
  [17] 9750
  [18] 9750
  [19] 9700

--- 2023 Schema ---
List file columns:
  [00] 1
  [01] 0235
  [02] nan
  [03] 3000837
  [04] MH
  [05] SURGERY AND PREMISES
  [06] 4159370000
  [07] MEDICAL CENTRE, ABBEYFIELDS, ELSTOW, BEDFORD
  [08] nan
  [09] MEDICAL CENTRE
  [10] ABBEYFIE

In [ ]:
# ==========================================
# PART 2 (COMMERCIAL): COMMERCIAL DATA SHATTERING & PANEL STACKING
# ==========================================
import pandas as pd
import numpy as np
import gc
import json

print("Initiating Memory-Efficient VOA Data Shattering (Panel Approach)...")

# Indices based on confirmed schema
cols_list_idx   = [19, 14, 5, 17]
cols_list_names = ['Assessment_UARN', 'Postcode', 'Property_Sector', 'Rateable_Value']
cols_sum_idx    = [0, 1, 16]
cols_sum_names  = ['Line_Number', 'Assessment_UARN', 'Total_Floor_Area']
devon_prefixes  = ('EX', 'TQ', 'PL')

def process_voa_epoch_fixed(list_path, sum_path, label):
    print(f"Processing {label}...")

    # Load Summary
    df_sum = pd.read_csv(sum_path, sep='*', header=None, usecols=cols_sum_idx,
                         names=cols_sum_names, dtype=str, low_memory=False)
    df_sum = df_sum[df_sum['Line_Number'] == '01'][['Assessment_UARN', 'Total_Floor_Area']]
    df_sum['Assessment_UARN'] = df_sum['Assessment_UARN'].str.strip()
    sum_dict = df_sum.set_index('Assessment_UARN')['Total_Floor_Area'].to_dict()
    del df_sum
    gc.collect()

    # Process List
    chunk_list = []
    for chunk in pd.read_csv(list_path, sep='*', header=None, chunksize=500000,
                             dtype=str, low_memory=False):
        chunk = chunk[cols_list_idx]
        chunk.columns = cols_list_names

        # Filter and Join
        chunk = chunk[chunk['Postcode'].str.startswith(devon_prefixes, na=False)]
        chunk['Assessment_UARN'] = chunk['Assessment_UARN'].str.strip()
        chunk['Total_Floor_Area'] = chunk['Assessment_UARN'].map(sum_dict)

        chunk = chunk.dropna(subset=['Total_Floor_Area'])
        chunk_list.append(chunk)

    df_final = pd.concat(chunk_list, ignore_index=True)
    df_final['Postcode'] = df_final['Postcode'].str.upper().str.strip()
    df_final['Total_Floor_Area'] = pd.to_numeric(df_final['Total_Floor_Area'], errors='coerce')
    df_final['Rateable_Value'] = pd.to_numeric(df_final['Rateable_Value'], errors='coerce')

    # Update Attrition Log with correct epoch key
    attrition_log[f"VOA_{label}"]["03_Final_Devon_Rows"] = len(df_final)

    return df_final

# Execute
df_2017 = process_voa_epoch_fixed(voa_2017_list_path, voa_2017_summary_path, "2017")
df_2023 = process_voa_epoch_fixed(voa_2023_list_path, voa_2023_summary_path, "2023")

# Construct the panel
df_2017['Year'] = 2017
df_2023['Year'] = 2023
df_commercial_panel = pd.concat([df_2017, df_2023], ignore_index=True)

# Capture initial panel size
initial_panel_size = len(df_commercial_panel)

# Identify rows with zero or negative values
zero_entities = df_commercial_panel[
    (df_commercial_panel['Rateable_Value'] <= 0) |
    (df_commercial_panel['Total_Floor_Area'] <= 0)
]

# Log them to the quality checks folder before dropping
if not zero_entities.empty:
    drop_log_path = os.path.join(PROJECT_ROOT, '05_logs/quality_checks/dropped_zero_values_commercial.csv')
    zero_entities.to_csv(drop_log_path, index=False)
    print(f"Flagged {len(zero_entities)} zero-value records. Saved to quality_checks/.")

# Apply the filter to keep only strictly positive values
df_commercial_panel = df_commercial_panel[
    (df_commercial_panel['Rateable_Value'] > 0) &
    (df_commercial_panel['Total_Floor_Area'] > 0)
].copy()

# Log Transformations for Econometric Modeling
df_commercial_panel['Log_Rateable_Value'] = np.log(df_commercial_panel['Rateable_Value'])
df_commercial_panel['Log_Floor_Area'] = np.log(df_commercial_panel['Total_Floor_Area'])

# Save to Parquet
df_commercial_panel.to_parquet(output_com_cleaned, index=False)

# Perfectly clean log entry accounting for dropped rows
attrition_log["VOA_Panel"] = {
    "04_Initial_Stacked_Panel": initial_panel_size,
    "05_Zero_Filtered_Panel": len(df_commercial_panel),
    "Note": "Combined 2017 and 2023 Devon commercial assessments (zero-values removed)"
}

with open(output_attrition_log, 'w') as f:
    json.dump(attrition_log, f, indent=4)

print(f"Cell 6 Complete: Panel created with {len(df_commercial_panel):,} commercial observations.")

Initiating Memory-Efficient VOA Data Shattering (Panel Approach)...
Processing 2017...
Processing 2023...
Flagged 8 zero-value records. Saved to quality_checks/.
Cell 6 Complete: Panel created with 124,797 commercial observations.


In [ ]:
# ==========================================
# PART 2 (COMMERCIAL) POST-CHECK: VERIFYING THE 124,805 COUNT
# ==========================================
print("--- 1. Split by Year ---")
print(df_commercial_panel['Year'].value_counts())

print("\n--- 2. Postcode Prefix Distribution (Ensuring they are Devon) ---")
# Check how many are EX, TQ, PL vs others
df_commercial_panel['Prefix'] = df_commercial_panel['Postcode'].str[:2]
print(df_commercial_panel['Prefix'].value_counts(normalize=True).head(10))

print("\n--- 3. Top 5 Property Sectors ---")
print(df_commercial_panel['Property_Sector'].value_counts().head(5))

# How many are duplicates vs unique assessment units?
dup_count = df_commercial_panel.duplicated(subset=['Assessment_UARN', 'Year']).sum()
print(f"\nDuplicates found within specific years: {dup_count}")

--- 1. Split by Year ---
Year
2017    63167
2023    61630
Name: count, dtype: int64

--- 2. Postcode Prefix Distribution (Ensuring they are Devon) ---
Prefix
EX    0.428832
PL    0.356827
TQ    0.214340
Name: proportion, dtype: float64

--- 3. Top 5 Property Sectors ---
Property_Sector
SELF CATERING HOLIDAY UNIT AND PREMISES    21117
SHOP AND PREMISES                          20763
OFFICES AND PREMISES                       13011
WORKSHOP AND PREMISES                      11564
STORE AND PREMISES                          6519
Name: count, dtype: int64

Duplicates found within specific years: 0


In [ ]:
# DIAGNOSTIC: Cross-year UARN matching
# Result: 0% match confirmed. 2017 and 2023 files use different
# column structures and identifier formats. Cross-list matching
# is not possible in this pipeline. See methodology section 3.7,
# Specification 4 caveat 3.

# Cross-year UARN matching diagnostic
uarns_2017 = set(df_commercial_panel[df_commercial_panel['Year']==2017]['Assessment_UARN'])
uarns_2023 = set(df_commercial_panel[df_commercial_panel['Year']==2023]['Assessment_UARN'])

matched = uarns_2017.intersection(uarns_2023)
only_2017 = uarns_2017 - uarns_2023
only_2023 = uarns_2023 - uarns_2017

print(f"\n=== CROSS-YEAR UARN MATCHING ===")
print(f"UARNs in 2017 list: {len(uarns_2017):,}")
print(f"UARNs in 2023 list: {len(uarns_2023):,}")
print(f"UARNs matched across both lists: {len(matched):,} ({len(matched)/len(uarns_2017)*100:.1f}% of 2017)")
print(f"UARNs only in 2017 (deleted/reconstituted): {len(only_2017):,}")
print(f"UARNs only in 2023 (new/split): {len(only_2023):,}")

# Check UARN format samples from each year
print("\n=== UARN FORMAT CHECK ===")
print("2017 UARN samples:")
print(df_commercial_panel[df_commercial_panel['Year']==2017]['Assessment_UARN'].head(10).tolist())
print("\n2023 UARN samples:")
print(df_commercial_panel[df_commercial_panel['Year']==2023]['Assessment_UARN'].head(10).tolist())


# Check raw column 19 values from each raw file directly
import pandas as pd

print("=== 2017 RAW COLUMN 19 SAMPLES ===")
df_check_2017 = next(pd.read_csv(voa_2017_list_path, sep='*', header=None,
                                   chunksize=5, dtype=str, low_memory=False))
print(df_check_2017[[19, 3]].head(5).to_string())

print("\n=== 2023 RAW COLUMN 19 SAMPLES ===")
df_check_2023 = next(pd.read_csv(voa_2023_list_path, sep='*', header=None,
                                   chunksize=5, dtype=str, low_memory=False))
print(df_check_2023[[19, 3]].head(5).to_string())


print("=== 2017 ALL COLUMNS FIRST ROW ===")
df_check_2017 = next(pd.read_csv(voa_2017_list_path, sep='*', header=None,
                                   chunksize=1, dtype=str, low_memory=False))
for i, val in enumerate(df_check_2017.iloc[0]):
    if val and len(str(val).strip()) > 3:
        print(f"  Col {i:02d}: {str(val).strip()}")

print("\n=== 2023 ALL COLUMNS FIRST ROW ===")
df_check_2023 = next(pd.read_csv(voa_2023_list_path, sep='*', header=None,
                                   chunksize=1, dtype=str, low_memory=False))
for i, val in enumerate(df_check_2023.iloc[0]):
    if val and len(str(val).strip()) > 3:
        print(f"  Col {i:02d}: {str(val).strip()}")








=== CROSS-YEAR UARN MATCHING ===
UARNs in 2017 list: 63,167
UARNs in 2023 list: 61,630
UARNs matched across both lists: 0 (0.0% of 2017)
UARNs only in 2017 (deleted/reconstituted): 63,167
UARNs only in 2023 (new/split): 61,630

=== UARN FORMAT CHECK ===
2017 UARN samples:
['19886710000', '23958036000', '24240608000', '23948210000', '23451466000', '22788279000', '18840671000', '16924383000', '16924382000', '18840666000']

2023 UARN samples:
['28128901000', '26207498000', '26851023000', '26207497000', '26072455000', '26851005000', '27026889000', '26071162000', '26071161000', '24296184000']
=== 2017 RAW COLUMN 19 SAMPLES ===
            19              3 
0  18427888000  00012434001011
1  18870894000  00015999010012
2  18870924000  00015999001001
3  18870972000  00015999005006
4  18870888000  00015999006009

=== 2023 RAW COLUMN 19 SAMPLES ===
            19       3 
0  24014654000  3000837
1  26452581000  3000449
2  26911170000  3001255
3  26911146000  1077209
4  26978656000  1132714
===

In [ ]:
# ==========================================
# PART 2 (COMMERCIAL) POST-CHECK: VOA PARQUET DATA INSPECTION
# ==========================================

# Use the full path from the root directory
file_path = os.path.join(PROJECT_ROOT, '02_processed_data/01_cleaned/commercial_cleaned.parquet')

# Load the file
df_check = pd.read_parquet(file_path)

# Verify success
print(f"Successfully loaded {len(df_check)} rows.")

import pandas as pd

# Print shape (rows, columns)
print(f"Total Dimensions: {df_check.shape[0]} rows and {df_check.shape[1]} columns.")

# Display the first 5 rows to check structure
print("\nFirst 5 rows of the dataset:")
print(df_check.head())

# Check column data types
print("\nColumn Data Types:")
print(df_check.dtypes)

print(df_check[['Rateable_Value', 'Total_Floor_Area', 'Log_Rateable_Value', 'Log_Floor_Area']].describe())

Successfully loaded 124797 rows.
Total Dimensions: 124797 rows and 8 columns.

First 5 rows of the dataset:
  Assessment_UARN  Postcode                 Property_Sector  Rateable_Value  \
0     19886710000  PL13 2RB           WORKSHOP AND PREMISES            9600   
1     23958036000  PL25 4QW            OFFICES AND PREMISES            5700   
2     24240608000  PL25 4QW            OFFICES AND PREMISES            1525   
3     23948210000  PL25 4QW            OFFICES AND PREMISES            4950   
4     23451466000  PL25 4QW  CAR PARKING SPACE AND PREMISES             800   

   Total_Floor_Area  Year  Log_Rateable_Value  Log_Floor_Area  
0            259.16  2017            9.169518        5.557446  
1             48.27  2017            8.648221        3.876810  
2             10.22  2017            7.329750        2.324347  
3             67.68  2017            8.507143        4.214791  
4              2.00  2017            6.684612        0.693147  

Column Data Types:
Assessment_UA